# TBsim scenario notebook

This notebook was generated automatically from a research document. The **code is a fixed, tested template** built on
[TBsim](https://github.com/starsimhub/tbsim) (an agent-based TB model on [Starsim](https://starsim.org)); the **parameters,
scenarios and narrative** were extracted from the research document by an AI step and are loaded in the *Configuration* section.

TBsim uses the LSHTM *spectrum of TB disease* natural history: susceptible → infection (or cleared) → non-infectious →
asymptomatic → symptomatic → treatment / cleared / dead.

> **Note:** This is a demonstration model. Parameters marked *placeholder* or *TBsim default* were not supplied by the
> research document. Results are not validated for research or policy decisions.

**How to run:** In Colab choose *Runtime → Run all*. The first cell installs TBsim (about 1–2 minutes).

In [ ]:
# Install TBsim from GitHub (skip if it is already installed)
import importlib.util, subprocess, sys
if importlib.util.find_spec('tbsim') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/starsimhub/tbsim.git'], check=True)

import base64, json, re, warnings
import numpy as np
import pandas as pd
import sciris as sc
import starsim as ss
import tbsim
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
print('tbsim', tbsim.__version__, '| starsim', ss.__version__)

## 1. Configuration from the research document

In [ ]:
# AI-extracted configuration (base64-encoded JSON, injected by the workflow)
CONFIG_B64 = "ewogICJ0aXRsZSI6ICJUQnNpbSBmaXhlZC1wYXJhbWV0ZXIgbm90ZWJvb2sgdGVtcGxhdGUgKFN0YXJzaW0gQUJNKSAtIFRCLWZvY3VzZWQgYXBwbGljYXRpb24iLAogICJzdW1tYXJ5X21kIjogIlRoaXMgbm90ZWJvb2sgdGVtcGxhdGUgYW5jaG9ycyBhIGZpeGVkLXBhcmFtZXRlciBUQiBzY2VuYXJpbyB1c2luZyB0aGUgU3RhcnNpbSBhZ2VudC1iYXNlZCBmcmFtZXdvcmsuIFN0YXJzaW0gaXMgZGVzaWduZWQgdG8gc2ltdWxhdGUgbXVsdGlwbGUgZGlzZWFzZXMsIHRyYW5zbWlzc2lvbiBuZXR3b3JrcywgZGVtb2dyYXBoaWNzLCBhbmQgaW50ZXJ2ZW50aW9ucyB3aXRoaW4gYSBzaW5nbGUgQUJNLCB3aXRoIGV4cGxpY2l0IHN1cHBvcnQgZm9yIHNlcGFyYXRlIHJhbmRvbS1udW1iZXIgc3RyZWFtcyB0byBtaW5pbWl6ZSBzdG9jaGFzdGljIG5vaXNlIGFjcm9zcyBpdGVyYXRpb25zLiBUaGUgVEItZm9jdXNlZCBjb25maWd1cmF0aW9uIGxldmVyYWdlcyBTdGFyc2ltJ3MgY2FwYWJpbGl0eSB0byBtb2RlbCByZXNwaXJhdG9yeSB0cmFuc21pc3Npb24sIGRpc2Vhc2Ugc3RhdGVzLCBhbmQgaW50ZXJ2ZW50aW9uIGNvbXBvbmVudHMgc3VjaCBhcyBkaWFnbm9zdGljcyBhbmQgdHJlYXRtZW50cywgd2hpbGUgYXZvaWRpbmcgY29tb3JiaWRpdHkgbW9kZWxpbmcgKGUuZy4sIEhJViwgbWFsbnV0cml0aW9uKSBiZXlvbmQgdGhlIFRCIGNvbnRleHQuIFRoZSB0ZW1wbGF0ZSBlbXBoYXNpemVzIGEgY29tcGFjdCBwb3B1bGF0aW9uIHdoZXJlIGFnZW50cyBpbnRlcmFjdCB3aXRoaW4gYSBuZXR3b3JrIHRoYXQgY2FuIHJlZmxlY3QgaG91c2Vob2xkLCBjb21tdW5pdHksIGFuZCByYW5kb20gY29udGFjdHMsIHdpdGggeWVhcmx5IHRpbWUgc3RlcHMgYW5kIHRoZSBwb3NzaWJpbGl0eSBvZiBmcmFjdGlvbmFsIGR0IHN0ZXBzIGlmIG5lZWRlZC4gSW50ZXJ2ZW50aW9ucyBjYW4gYmUgcmVwcmVzZW50ZWQgdGhyb3VnaCB0ZXN0aW5nIChkaWFnbm9zdGljIHBhdGh3YXlzKSwgdHJlYXRtZW50IHJlZ2ltZW5zLCBhbmQgcG90ZW50aWFsIHZhY2NpbmF0aW9uLWxpa2Ugc3RyYXRlZ2llcy4gVGhlIHN1bW1hcnkgbm90ZXMgdGhhdCBUQiBpbiB0aGlzIGNvbnRleHQgaXMgdHJlYXRlZCBhcyB0aGUgcHJpbWFyeSBwYXRob2dlbiBvZiBpbnRlcmVzdCwgd2l0aCB0aGUgbW9kZWwgY2FwYWJsZSBvZiBjYXB0dXJpbmcgaW5mZWN0aW9uIHNwcmVhZCwgcHJvZ3Jlc3Npb24sIGFuZCBvdXRjb21lcyB1bmRlciBhIGZpeGVkLXBhcmFtZXRlciBzZXR1cC4gTGltaXRhdGlvbnMgaW5jbHVkZSBwb3RlbnRpYWxseSBzaW1wbGlmaWVkIG5hdHVyYWwgaGlzdG9yeSBhc3N1bXB0aW9ucywgZml4ZWQgcGFyYW1ldGVycyBhY3Jvc3MgdGhlIHNpbXVsYXRpb24gaG9yaXpvbiwgYW5kIHRoZSBhYnNlbmNlIG9mIGNvLWluZmVjdGlvbiBvciBjb21vcmJpZGl0eSBkeW5hbWljcyB1bmxlc3MgZXhwbGljaXRseSBjb25maWd1cmVkIGluIHRoZSBUQi1mb2N1c2VkIFRCc2ltIG1vZHVsZS4iLAogICJpbnRlcnByZXRhdGlvbl9tZCI6ICJHdWlkYW5jZSBmb3IgcmVhZGluZyByZXN1bHRzOiB3aXRoIGZpeGVkIHBhcmFtZXRlcnMsIG9ic2VydmVkIHZhcmlhdGlvbiBhY3Jvc3MgcnVucyBwcmltYXJpbHkgcmVmbGVjdHMgc3RvY2hhc3RpYyBzYW1wbGluZyByYXRoZXIgdGhhbiBwYXJhbWV0ZXIgdW5jZXJ0YWludHkuIFJlc3VsdHMgc2hvdWxkIGJlIGludGVycHJldGVkIGFzIHByb2Nlc3MtY29uc2lzdGVudCB0cmFqZWN0b3JpZXMgdW5kZXIgdGhlIHNwZWNpZmllZCBUQiBzY2VuYXJpbywgbm90IGFzIHdlbGZhcmUgb3IgcG9saWN5IG9wdGltYWxpdHkgYWNyb3NzIHBhcmFtZXRlciBzcGFjZS4gVW5jZXJ0YWludHkgbWF5IGFyaXNlIGZyb20gc3RvY2hhc3RpYyBldmVudHMgaW4gdHJhbnNtaXNzaW9uIGFuZCBkaXNlYXNlIHByb2dyZXNzaW9uLCBuZXR3b3JrIGZvcm1hdGlvbiwgYW5kIHNhbXBsaW5nIG5vaXNlIGFjcm9zcyBpdGVyYXRpb25zLiBQbGFjZWhvbGRlciBlbGVtZW50cyBpbmNsdWRlIGFueSBvcHRpb25hbCBleHRlbnNpb25zIHN1Y2ggYXMgY28tbW9yYmlkaXRpZXMsIGNoYW5naW5nIHRyYW5zbWlzc2lvbiBwYXJhbWV0ZXJzIG92ZXIgdGltZSwgb3IgYWx0ZXJuYXRpdmUgZGlhZ25vc3RpYy90cmVhdG1lbnQgcGF0aHdheXMgdGhhdCBhcmUgbm90IGFjdGl2YXRlZCBpbiB0aGlzIGZpeGVkIGNvbmZpZ3VyYXRpb24uIFVzZSB0aGlzIHRlbXBsYXRlIHRvIGVzdGFibGlzaCBhIHJlcHJvZHVjaWJsZSBiYXNlbGluZSBiZWZvcmUgZXhwbG9yaW5nIHBhcmFtZXRlciBzd2VlcHMgb3IgYWRkaXRpb25hbCBpbnRlcnZlbnRpb25zLiIsCiAgInBhcmFtX3NvdXJjZXMiOiB7fQp9"

DEFAULTS = {
    "title": "TBsim scenario analysis",
    "summary_md": "_No research summary was provided._",
    "interpretation_md": "_No interpretation was provided._",
    "sim": {"n_agents": 3000, "start_year": 2000, "stop_year": 2025, "rand_seed": 1,
            "beta_per_year": 0.2, "init_prev": 0.05, "n_households": 600, "contacts_per_person": 5},
    "cascade": {"care_seeking_rate_per_day": 0.1, "screen_test": "CAD", "screen_coverage": 0.9,
                "confirm_test": "Xpert", "confirm_coverage": 0.8, "treatment": "DOTS"},
    "prevention": {"bcg_coverage": 0.8, "bcg_age_max": 5, "tpt_coverage": 0.8,
                   "beta_change_year": 2010, "beta_change_factor": 0.7},
    "resistance": {"include": True, "drug": "RIF", "rel_fitness": 0.9, "base_efficacy": 0.8,
                   "resist_penalty": 0.2, "init_prev": 0.12, "treatment_rate_per_year": 1.0},
    "scenarios": {"Baseline": [], "Care cascade": ["cascade"], "Cascade + prevention": ["cascade", "prevention"]},
    "param_sources": {},
}

def load_config(b64):
    """Decode the injected config; fall back to defaults for anything missing or invalid."""
    try:
        raw = base64.b64decode(b64).decode('utf-8', errors='replace')
        raw = raw[raw.find('{'): raw.rfind('}') + 1]   # strip any text or code fences around the JSON
        user = json.loads(raw)
    except Exception as e:
        print(f'WARNING: could not read the AI configuration ({e}); using TBsim template defaults.')
        user = {}
    cfg = sc.dcp(DEFAULTS)
    for k, v in user.items():
        if isinstance(v, dict) and isinstance(cfg.get(k), dict) and k not in ('scenarios', 'param_sources'):
            cfg[k].update(v)
        else:
            cfg[k] = v
    return cfg

def clamp(x, lo, hi, default):
    try:
        return float(min(max(float(x), lo), hi))
    except Exception:
        return default

cfg = load_config(CONFIG_B64)
S, C, P = cfg['sim'], cfg['cascade'], cfg['prevention']
R = cfg['resistance']

# Validate ranges so a bad value cannot crash the notebook
S['n_agents'] = int(clamp(S['n_agents'], 500, 20000, 3000))
S['start_year'] = int(clamp(S['start_year'], 1900, 2100, 2000))
S['stop_year'] = int(clamp(S['stop_year'], S['start_year'] + 2, S['start_year'] + 60, S['start_year'] + 25))
for g, keys in [('sim', ['init_prev']), ('cascade', ['screen_coverage', 'confirm_coverage']),
                ('prevention', ['bcg_coverage', 'tpt_coverage']),
                ('resistance', ['rel_fitness', 'base_efficacy', 'resist_penalty', 'init_prev'])]:
    for k in keys:
        cfg[g][k] = clamp(cfg[g][k], 0, 1, DEFAULTS[g][k])   # probabilities/coverages must be in [0, 1]
for g, k, lo, hi in [('sim', 'beta_per_year', 0, 50), ('sim', 'n_households', 10, 5000), ('sim', 'contacts_per_person', 0, 50),
                     ('cascade', 'care_seeking_rate_per_day', 0, 1), ('prevention', 'bcg_age_max', 0, 100),
                     ('prevention', 'beta_change_factor', 0, 10), ('resistance', 'treatment_rate_per_year', 0, 50)]:
    cfg[g][k] = clamp(cfg[g][k], lo, hi, DEFAULTS[g][k])
P['beta_change_year'] = int(clamp(P['beta_change_year'], S['start_year'], S['stop_year'], S['start_year'] + 10))
if not isinstance(cfg['scenarios'], dict) or not cfg['scenarios']:
    cfg['scenarios'] = DEFAULTS['scenarios']
if not isinstance(cfg['param_sources'], dict):
    cfg['param_sources'] = {}
DX = {'CAD': tbsim.CAD, 'Xpert': tbsim.Xpert, 'OralSwab': tbsim.OralSwab, 'FujiLAM': tbsim.FujiLAM}
TX = {'DOTS': tbsim.DOTS, 'DOTSImproved': tbsim.DOTSImproved, 'FirstLine': tbsim.FirstLine, 'SecondLine': tbsim.SecondLine}
C['screen_test'] = C['screen_test'] if C['screen_test'] in DX else 'CAD'
C['confirm_test'] = C['confirm_test'] if C['confirm_test'] in DX else 'Xpert'
C['treatment'] = C['treatment'] if C['treatment'] in TX else 'DOTS'

display(Markdown(f"# {cfg['title']}\n\n{cfg['summary_md']}"))

# Parameter table with sources
rows = []
for group in ['sim', 'cascade', 'prevention', 'resistance']:
    for k, v in cfg[group].items():
        rows.append({'group': group, 'parameter': k, 'value': v,
                     'source': cfg['param_sources'].get(f'{group}.{k}', cfg['param_sources'].get(k, 'TBsim template default'))})
pd.set_option('display.max_rows', 200)
display(pd.DataFrame(rows))

## 2. Model builders
Every scenario gets fresh module instances, so the same object is never reused across simulations.

In [ ]:
def base_pars():
    return dict(n_agents=S['n_agents'], start=ss.date(f"{S['start_year']}-01-01"), stop=ss.date(f"{S['stop_year']}-12-31"),
                dt=ss.days(7), rand_seed=int(S['rand_seed']), verbose=0,
                beta=ss.peryear(float(S['beta_per_year'])), init_prev=ss.bernoulli(S['init_prev']))

def make_networks():
    """Random community contacts plus a static household network (needed for household TPT)."""
    rng = np.random.default_rng(int(S['rand_seed']))
    n = int(S['n_households'])
    ages = [sc.strjoin(rng.integers(1, 70, rng.integers(2, 6))) for _ in range(n)]
    hh = ss.library.HouseholdNet(dhs_data=sc.dataframe(hh_id=np.arange(n), ages=ages), dynamic=False)
    return [ss.RandomNet(pars=dict(n_contacts=ss.poisson(lam=float(S['contacts_per_person'])), dur=0)), hh]

def cascade_interventions():
    return [tbsim.HealthSeekingBehavior(pars=dict(initial_care_seeking_rate=ss.perday(float(C['care_seeking_rate_per_day'])))),
            tbsim.DxDelivery(name='screen', product=DX[C['screen_test']](), coverage=C['screen_coverage'], result_state='screen_positive'),
            tbsim.DxDelivery(name='confirm', product=DX[C['confirm_test']](), coverage=C['confirm_coverage'],
                             eligibility=lambda sim: sim.people.screen.screen_positive.uids, result_state='diagnosed'),
            tbsim.TxDelivery(product=TX[C['treatment']]())]

def prevention_interventions():
    return [tbsim.BCGRoutine(pars=dict(coverage=P['bcg_coverage'], age_range=[0, float(P['bcg_age_max'])])),
            tbsim.TPTHousehold(pars=dict(coverage=P['tpt_coverage'])),
            tbsim.BetaByYear(pars=dict(years=[int(P['beta_change_year'])], x_beta=[float(P['beta_change_factor'])]))]

def build_sim(components=(), analyzers=None, label=''):
    """components: any of 'cascade', 'prevention'."""
    intvs, kw = [], {}
    if 'cascade' in components: intvs += cascade_interventions()
    if 'prevention' in components: intvs += prevention_interventions()
    return tbsim.Sim(**base_pars(), networks=make_networks(), interventions=intvs or None, analyzers=analyzers, label=label, **kw)

def series(sim, key):
    return np.asarray(sim.results.flatten()[key].values, dtype=float)

def years(sim):
    return np.array([t.years if hasattr(t, 'years') else float(t) for t in sim.timevec])

def plot_keys(sim, keys, title):
    fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.5))
    for ax, k in zip(np.atleast_1d(axes), keys):
        ax.plot(years(sim), series(sim, k)); ax.set_title(k); ax.set_xlabel('Year')
    fig.suptitle(title); plt.tight_layout(); plt.show()

## 3. Baseline natural history
No interventions: TB spreads through community and household contacts and progresses through the disease spectrum.

In [ ]:
sim_base = build_sim(label='Baseline'); sim_base.run()
plot_keys(sim_base, ['tb_prevalence_active', 'tb_incidence_kpy', 'tb_new_deaths'], 'Baseline')
states = ['tb_n_infection', 'tb_n_non_infectious', 'tb_n_asymptomatic', 'tb_n_symptomatic', 'tb_n_treatment']
plt.figure(figsize=(9, 4))
for k in states: plt.plot(years(sim_base), series(sim_base, k), label=k.replace('tb_n_', ''))
plt.legend(); plt.title('Agents in each TB state (baseline)'); plt.xlabel('Year'); plt.show()

## 4. Care cascade
Health-seeking behaviour → screening → confirmatory test → treatment, using the tests, coverages and regimen from the configuration.

In [ ]:
sim_cascade = build_sim(['cascade'], analyzers=[tbsim.DwtAnalyzer(scenario_name='cascade')], label='Care cascade'); sim_cascade.run()
r = sim_cascade.results
print(f"Screened: {r.screen.n_tested.sum():,.0f} | Confirmatory tests: {r.confirm.n_tested.sum():,.0f}")
plot_keys(sim_cascade, ['tb_prevalence_active', 'tb_n_treatment', 'tb_new_deaths'], 'Care cascade')

## 5. Prevention
BCG vaccination of young children, TPT for household contacts of people starting treatment (on top of the cascade),
and a step change in transmission (BetaByYear).

In [ ]:
sim_prev = build_sim(['cascade', 'prevention'], label='Cascade + prevention'); sim_prev.run()
plot_keys(sim_prev, ['tb_prevalence_active', 'tb_incidence_kpy', 'tb_new_infections'], 'Cascade + prevention')

## 6. Drug resistance
`TBResistant` tracks drug-susceptible and resistant strains; strain-aware treatment (`TxR`) has reduced efficacy against
resistant strains. `ResistanceStats` reports the resistant fraction and where new resistance comes from.

In [ ]:
if R.get('include'):
    drug = str(R.get('drug', 'RIF')) or 'RIF'
    tbr = tbsim.TBResistant(drugs=[drug], rel_fitness={drug: R['rel_fitness']}, beta=ss.permonth(0.35), init_prev=ss.bernoulli(R['init_prev']))
    tx = tbsim.TxDeliveryR(rate_sym=ss.peryear(float(R['treatment_rate_per_year'])),
                           product=tbsim.TxR(strains=tbr.strains, base_efficacy=R['base_efficacy'], resist_penalty={drug: R['resist_penalty']}))
    sim_res = tbsim.Sim(tb_model=tbr, n_agents=S['n_agents'], demographics=[], networks=ss.RandomNet(pars=dict(n_contacts=ss.poisson(lam=8), dur=0)),
                        interventions=[tx], analyzers=[tbsim.ResistanceStats()], dt=ss.days(30),
                        start=ss.date(f"{S['start_year']}-01-01"), stop=ss.date(f"{S['stop_year']}-12-31"), rand_seed=int(S['rand_seed']), verbose=0)
    sim_res.run()
    flat = sim_res.results.flatten()
    keys = [k for k in flat.keys() if 'frac_resist' in k or 'flux_' in k][:4]
    fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 3.5))
    for ax, k in zip(np.atleast_1d(axes), keys):
        ax.plot(years(sim_res), np.asarray(flat[k].values, dtype=float)); ax.set_title(k); ax.set_xlabel('Year')
    fig.suptitle(f'Drug resistance ({drug})'); plt.tight_layout(); plt.show()
else:
    sim_res = None
    print('Drug resistance is turned off in the configuration.')

## 7. Scenario comparison
Scenarios defined in the configuration. Each scenario lists components: `cascade`, `prevention`.

In [ ]:
VALID = {'cascade', 'prevention'}
scen_sims = {}
for name, comps in dict(cfg['scenarios']).items():
    comps = [c for c in (comps or []) if c in VALID]
    s = build_sim(comps, label=str(name)); s.run(); scen_sims[str(name)] = s

tbsim.plot({n: s.results.flatten() for n, s in scen_sims.items()},
           select=dict(like=['prevalence_active', 'incidence_kpy', 'new_deaths']), n_cols=3)

summary = pd.DataFrame([{
    'scenario': n,
    'final active prevalence': series(s, 'tb_prevalence_active')[-1],
    'cumulative infections': series(s, 'tb_cum_infections')[-1],
    'total TB deaths': series(s, 'tb_new_deaths').sum(),
} for n, s in scen_sims.items()])
display(summary.round(4))

## 8. Dwell-time analysis
Time agents spend in each TB state (from the care-cascade run).

In [ ]:
dwt = [a for a in sim_cascade.analyzers.values() if isinstance(a, tbsim.DwtAnalyzer)][0]
try:
    dwt.plot('histogram')
    plt.show()
except Exception as e:
    print('Dwell-time plot unavailable:', e)

## 9. Sanity checks

In [ ]:
all_sims = [sim_base, sim_cascade, sim_prev] + list(scen_sims.values())
for s in all_sims:
    prev = series(s, 'tb_prevalence')
    assert np.all(np.isfinite(prev)), f'{s.label}: non-finite prevalence'
    assert np.all((prev >= 0) & (prev <= 1)), f'{s.label}: prevalence outside [0, 1]'
    for k in ['tb_new_infections', 'tb_new_deaths', 'tb_n_symptomatic']:
        assert np.all(series(s, k) >= 0), f'{s.label}: negative {k}'
print(f'All checks passed for {len(all_sims)} simulations.')

## 10. Interpretation and limitations

In [ ]:
display(Markdown(cfg['interpretation_md']))
display(Markdown("_This notebook uses a fixed TBsim template. Values marked as template defaults or placeholders were not supplied by the research document; results are illustrative, not validated._"))